# 04. 모델링 — RevPAR 예측 (LightGBM + SHAP)

**분석 목적:** 호스트가 통제 가능한 변수로 RevPAR를 예측하는 모델을 구축하고, 각 변수의 영향력을 정량화합니다.

**핵심 질문:**
- 어떤 호스트 행동 변수가 RevPAR에 가장 큰 영향을 주는가?
- 모델이 실제 운영 의사결정에 활용 가능한 수준인가?

**접근 방법:**
- LightGBM · RandomForest · Ridge 비교 (5-Fold CV)
- SHAP으로 피처 기여도 분해 → 호스트 액션 우선순위 도출

**데이터:** Active+Operating 서브셋 (14,399개) → train/test 8:2 분할  
**모델:** LightGBM · RandomForest · Ridge · DummyRegressor(baseline)

In [ ]:
# RANDOM_SEED=42 고정 -- 5-Fold CV와 모델 학습의 재현성 보장
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

# 재현성 시드
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# 데이터 경로
DATA_DIR   = Path('../data/processed')
REPORT_DIR = Path('../reports')

# 도메인
DOMAIN_KOREAN   = '서울 에어비앤비'
TARGET_COL      = 'log_target'   # log1p(ttm_revpar)
REVPAR_MAX      = 2_000_000      # 예측값 합리성 검증 상한

# 시각화
sns.set_style('whitegrid')
import platform
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
else:
    plt.rc('font', family='AppleGothic')
plt.rcParams['axes.unicode_minus'] = False
PALETTE = 'Set2'

In [ ]:
# review_rate 제거 -- ttm_occupancy의 인코딩 형태로 누수 확정 (노트북 07 참조)
X_train = pd.read_csv(DATA_DIR / 'X_train_host.csv', index_col='Unnamed: 0')
X_test  = pd.read_csv(DATA_DIR / 'X_test_host.csv',  index_col='Unnamed: 0')
y_train = pd.read_csv(DATA_DIR / 'y_train_host_log.csv', index_col='Unnamed: 0')[TARGET_COL]
y_test  = pd.read_csv(DATA_DIR / 'y_test_host_log.csv',  index_col='Unnamed: 0')[TARGET_COL]
y_train_orig = pd.read_csv(DATA_DIR / 'y_train_host_orig.csv', index_col='Unnamed: 0').squeeze()
y_test_orig  = pd.read_csv(DATA_DIR / 'y_test_host_orig.csv',  index_col='Unnamed: 0').squeeze()

# bool 컬럼을 int로 변환 (LightGBM 호환)
bool_cols = X_train.select_dtypes('bool').columns
X_train[bool_cols] = X_train[bool_cols].astype(int)
X_test[bool_cols]  = X_test[bool_cols].astype(int)

# ── review_rate 제거 (ttm_occupancy 인코딩 누수 확정 — notebook 07 Cell 8 참조) ──
X_train = X_train.drop(columns=['review_rate'])
X_test  = X_test.drop(columns=['review_rate'])

print(X_train.columns.tolist())

In [ ]:
# Ridge는 스케일에 민감 -> RobustScaler 사용 (이상치에 강건), 트리 모델은 스케일링 불필요
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer

# 수치형 연속 컬럼 (RobustScaler — 이상치에 강건)
# review_rate 제거됨 (ttm_occupancy 누수 변수)
numeric_cols = [
    'min_nights', 'rating_overall', 'photos_count', 'num_reviews',
    'nearest_poi_dist_km', 'district_listing_count',
    'district_superhost_rate', 'district_operating_rate',
    'district_entire_home_rate', 'ttm_pop'
]

# 이진/순서형 컬럼 (스케일링 불필요 — 트리 모델과 공유)
passthrough_cols = [c for c in X_train.columns if c not in numeric_cols]

preprocessor = ColumnTransformer([
    ('scaler', RobustScaler(), numeric_cols),
    ('pass',   'passthrough',  passthrough_cols)
], remainder='drop')

# 스케일러 fit은 X_train에서만 수행 (데이터 누수 방지)
X_train_scaled = preprocessor.fit_transform(X_train)
X_test_scaled  = preprocessor.transform(X_test)

In [ ]:
# DummyRegressor(중앙값)를 베이스라인으로 -- 이보다 낮은 모델은 의미 없음
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
from lightgbm import LGBMRegressor

models = {
    'DummyRegressor (Baseline)': DummyRegressor(strategy='median'),
    'Ridge': Ridge(alpha=1.0, random_state=RANDOM_SEED),
    'RandomForest': RandomForestRegressor(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=10,
        random_state=RANDOM_SEED,
        n_jobs=-1
    ),
    'LightGBM': LGBMRegressor(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=63,
        min_child_samples=30,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        random_state=RANDOM_SEED,
        n_jobs=-1,
        verbosity=-1
    )
}

# 트리 기반 모델은 DataFrame(컬럼명 보존), Ridge는 스케일된 피처 사용
X_for_model = {
    'DummyRegressor (Baseline)': (X_train.values, X_test.values),
    'Ridge':                     (X_train_scaled,  X_test_scaled),
    'RandomForest':              (X_train,          X_test),
    'LightGBM':                  (X_train,          X_test),
}

print('모델 정의 완료:')
for name in models:
    print(f'  {name}')

In [ ]:
# 5-Fold CV -- 단순 train/test 분할보다 분산 추정이 안정적
from sklearn.model_selection import KFold, cross_validate
from sklearn.metrics import make_scorer, r2_score, mean_absolute_error, mean_squared_error

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

def rmse_score(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

scoring = {
    'R2':   make_scorer(r2_score),
    'MAE':  make_scorer(mean_absolute_error, greater_is_better=False),
    'RMSE': make_scorer(rmse_score, greater_is_better=False)
}

cv_results = {}

for name, model in models.items():
    X_tr, _ = X_for_model[name]
    scores = cross_validate(model, X_tr, y_train, cv=cv, scoring=scoring,
                            return_train_score=True, n_jobs=-1)
    cv_results[name] = {
        'Train R2':   scores['train_R2'].mean(),
        'CV R2':      scores['test_R2'].mean(),
        'CV R2 std':  scores['test_R2'].std(),
        'CV MAE':     -scores['test_MAE'].mean(),
        'CV RMSE':    -scores['test_RMSE'].mean(),
    }
    print(f"✅ {name:<35} CV R² = {cv_results[name]['CV R2']:.4f} ± {cv_results[name]['CV R2 std']:.4f}")

print('\n5-Fold CV 완료')

In [ ]:
# 모델 비교 시각화 -- R2 오차 막대(std)로 모델 안정성도 함께 확인
cv_df = pd.DataFrame(cv_results).T
cv_df = cv_df.round(4)
cv_df.index.name = '모델'

print('=' * 60)
print('5-Fold CV 성능 비교 (log1p 스케일)')
print('=' * 60)
print(cv_df[['Train R2', 'CV R2', 'CV R2 std', 'CV MAE', 'CV RMSE']].to_string())
print('=' * 60)

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

model_names = list(cv_results.keys())
r2_vals  = [cv_results[m]['CV R2']   for m in model_names]
mae_vals = [cv_results[m]['CV MAE']  for m in model_names]
r2_std   = [cv_results[m]['CV R2 std'] for m in model_names]

colors = sns.color_palette(PALETTE, len(model_names))

axes[0].barh(model_names, r2_vals, xerr=r2_std, color=colors, alpha=0.85, capsize=4)
axes[0].set_xlabel('CV R²')
axes[0].set_title('5-Fold CV R² (log 스케일)')
axes[0].axvline(0, color='gray', linestyle='--', linewidth=0.8)
for i, v in enumerate(r2_vals):
    axes[0].text(v + 0.005, i, f'{v:.4f}', va='center', fontsize=9)

axes[1].barh(model_names, mae_vals, color=colors, alpha=0.85)
axes[1].set_xlabel('CV MAE (log 스케일)')
axes[1].set_title('5-Fold CV MAE (낮을수록 좋음)')
for i, v in enumerate(mae_vals):
    axes[1].text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=9)

plt.suptitle('호스트 관점 RevPAR 예측 — 모델 CV 성능 비교', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print('저장: reports/fig_03_cv_comparison.png')

In [ ]:
# log 스케일 예측 후 expm1으로 역변환 -- 원 스케일 MAE를 원화(KRW)로 해석 가능
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

def mean_absolute_percentage_error(y_true, y_pred):
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

test_results = {}

fitted_models = {}  # SHAP용 모델 저장

for name, model in models.items():
    X_tr, X_te = X_for_model[name]
    model.fit(X_tr, y_train)
    fitted_models[name] = model

    # log 스케일 예측
    y_pred_log = model.predict(X_te)

    # 원 스케일 역변환
    y_pred_orig = np.expm1(y_pred_log)
    y_true_orig = y_test_orig.values

    # 예측값 범위 검증 (도메인 합리성)
    out_of_range = ((y_pred_orig < 0) | (y_pred_orig > REVPAR_MAX)).sum()
    y_pred_orig_clipped = np.clip(y_pred_orig, 0, REVPAR_MAX)

    test_results[name] = {
        'Test R2 (log)':    r2_score(y_test, y_pred_log),
        'Test MAE (원본₩)':  mean_absolute_error(y_true_orig, y_pred_orig_clipped),
        'Test RMSE (원본₩)': np.sqrt(mean_squared_error(y_true_orig, y_pred_orig_clipped)),
        'Test MAPE (%)':    mean_absolute_percentage_error(y_true_orig, y_pred_orig_clipped),
        '범위 이탈 예측 수':   int(out_of_range),
    }

test_df = pd.DataFrame(test_results).T.round(2)
test_df.index.name = '모델'
print('=' * 70)
print('최종 Test Set 성능 (원본 RevPAR 스케일, ₩)')
print('=' * 70)
print(test_df.to_string())
print('=' * 70)

In [ ]:
# log/원본 스케일 동시 시각화 -- 스케일 변환 전후 성능 차이를 직관적으로 확인
best_model_name = 'LightGBM'
X_tr_best, X_te_best = X_for_model[best_model_name]
lgbm_model = fitted_models[best_model_name]

y_pred_log_lgbm  = lgbm_model.predict(X_te_best)
y_pred_orig_lgbm = np.clip(np.expm1(y_pred_log_lgbm), 0, REVPAR_MAX)
y_true_orig_arr  = y_test_orig.values

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: log 스케일 산점도
axes[0].scatter(y_test, y_pred_log_lgbm, alpha=0.2, s=5, color='steelblue')
lims = [min(y_test.min(), y_pred_log_lgbm.min()),
        max(y_test.max(), y_pred_log_lgbm.max())]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='완벽한 예측선')
axes[0].set_xlabel('실제 log1p(RevPAR)')
axes[0].set_ylabel('예측 log1p(RevPAR)')
axes[0].set_title(f'LightGBM — log 스케일\nR² = {test_results[best_model_name]["Test R2 (log)"]:.4f}')
axes[0].legend(fontsize=8)

# 오른쪽: 원본 스케일 (최대 ₩500K 범위 확대)
mask_vis = y_true_orig_arr <= 500_000
axes[1].scatter(y_true_orig_arr[mask_vis], y_pred_orig_lgbm[mask_vis], alpha=0.2, s=5, color='seagreen')
lim2 = [0, 500_000]
axes[1].plot(lim2, lim2, 'r--', linewidth=1.5, label='완벽한 예측선')
axes[1].set_xlabel('실제 RevPAR (₩)')
axes[1].set_ylabel('예측 RevPAR (₩)')
axes[1].set_title(f'LightGBM — 원본 스케일 (₩500K 이하)\nMAE = ₩{test_results[best_model_name]["Test MAE (원본₩)"]:,.0f}')
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'₩{x/1000:.0f}K'))
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'₩{x/1000:.0f}K'))
axes[1].legend(fontsize=8)

plt.suptitle('LightGBM — 예측 vs 실제 RevPAR', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print('저장: reports/fig_03_pred_vs_actual.png')

In [ ]:
# 잔차 분석 -- 이분산성(heteroscedasticity) 존재 여부 확인
residuals = y_test.values - y_pred_log_lgbm

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 잔차 히스토그램
axes[0].hist(residuals, bins=60, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_xlabel('잔차 (실제 - 예측, log 스케일)')
axes[0].set_ylabel('빈도')
axes[0].set_title(f'잔차 분포\n평균={residuals.mean():.3f}, 표준편차={residuals.std():.3f}')

# 예측값 vs 잔차
axes[1].scatter(y_pred_log_lgbm, residuals, alpha=0.2, s=5, color='darkorange')
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('예측값 (log 스케일)')
axes[1].set_ylabel('잔차')
axes[1].set_title('예측값 vs 잔차 (이분산성 확인)')

plt.suptitle('LightGBM — 잔차 분석', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print('저장: reports/fig_03_residuals.png')

## SHAP 분석 — 모델 해석

LightGBM 예측에서 각 피처가 RevPAR에 미치는 영향을 정량화합니다.  
**TreeExplainer** 사용 (트리 모델 전용, 빠르고 정확).

In [ ]:
# TreeExplainer 사용 -- 트리 모델 전용으로 빠르고 정확한 SHAP 계산
import shap

N_SHAP = min(500, len(X_test))
np.random.seed(RANDOM_SEED)
shap_idx = np.random.choice(len(X_test), N_SHAP, replace=False)
X_shap = X_test.iloc[shap_idx].values
y_shap_orig = y_test_orig.iloc[shap_idx].values

explainer   = shap.TreeExplainer(lgbm_model)
shap_values = explainer.shap_values(X_shap)

feature_names = X_train.columns.tolist()

print(f'SHAP 계산 완료 — 샘플 수: {N_SHAP}')

In [ ]:
# Beeswarm Plot -- 피처별 영향 방향(+/-) 및 크기를 동시에 시각화
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values, X_shap,
    feature_names=feature_names,
    show=False,
    plot_size=None
)
plt.title('SHAP Summary Plot — RevPAR 예측 피처 영향 (LightGBM)', fontsize=12, pad=15)
plt.tight_layout()
plt.show()
print('저장: reports/fig_03_shap_summary.png')

In [ ]:
# 평균 절대 SHAP 값으로 순위화 -- 호스트가 통제 가능한 변수 우선순위 도출
mean_shap = np.abs(shap_values).mean(axis=0)
shap_importance = pd.Series(mean_shap, index=feature_names).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
colors_bar = sns.color_palette(PALETTE, len(shap_importance))
ax.barh(shap_importance.index[::-1], shap_importance.values[::-1], color=colors_bar[::-1])
ax.set_xlabel('평균 |SHAP 값| (log1p RevPAR 기여도)')
ax.set_title('SHAP 피처 중요도 — 호스트 통제 가능 변수 순위', fontsize=12)
ax.axvline(0, color='gray', linestyle='--', linewidth=0.8)
plt.tight_layout()
plt.show()

print('\n📊 SHAP 피처 중요도 전체 (28개):')
print(shap_importance.to_string())
print()
print('\n📊 SHAP 피처 중요도 TOP 10:')
for i, (feat, val) in enumerate(shap_importance.head(10).items(), 1):
    print(f'  {i:2d}. {feat:<40} {val:.4f}')
print('저장: reports/fig_03_shap_bar.png')

In [ ]:
# 상위 3 피처의 값 변화에 따른 SHAP 변화 -- 각 변수의 임계점 확인
top3_features = shap_importance.head(3).index.tolist()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for i, feat in enumerate(top3_features):
    feat_idx = feature_names.index(feat)
    axes[i].scatter(
        X_shap[:, feat_idx],
        shap_values[:, feat_idx],
        alpha=0.4, s=10,
        c=shap_values[:, feat_idx],
        cmap='RdYlBu_r'
    )
    axes[i].axhline(0, color='gray', linestyle='--', linewidth=0.8)
    axes[i].set_xlabel(feat, fontsize=9)
    axes[i].set_ylabel('SHAP 값' if i == 0 else '')
    axes[i].set_title(f'{feat}\nSHAP 의존도', fontsize=9)

plt.suptitle('SHAP Dependence Plot — 상위 3 피처', fontsize=12, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()
print('저장: reports/fig_03_shap_dependence.png')

In [ ]:
# 고/중/저 RevPAR 케이스 3개 -- 개별 예측의 기여 분해로 설명 가능성 제시
shap_pred_orig = np.expm1(lgbm_model.predict(X_shap))

# 고/중/저 각 1개씩 선택
idx_high = np.argmax(shap_pred_orig)                           # 최고 예측
idx_low  = np.argmin(shap_pred_orig)                           # 최저 예측
idx_mid  = np.argmin(np.abs(shap_pred_orig - np.median(shap_pred_orig)))  # 중앙값

cases = [
    ('고 RevPAR 케이스', idx_high),
    ('중간 RevPAR 케이스', idx_mid),
    ('저 RevPAR 케이스', idx_low)
]

fig, axes = plt.subplots(1, 3, figsize=(14, 6))

for ax, (title, idx) in zip(axes, cases):
    shap_vals_case = shap_values[idx]
    # 상위 8개 피처만 표시
    abs_order = np.argsort(np.abs(shap_vals_case))[::-1][:8]
    top_feats  = [feature_names[j] for j in abs_order]
    top_shap   = shap_vals_case[abs_order]
    colors_wf  = ['#d73027' if v > 0 else '#4575b4' for v in top_shap]

    ax.barh(range(len(top_feats)), top_shap[::-1], color=colors_wf[::-1])
    ax.set_yticks(range(len(top_feats)))
    ax.set_yticklabels(top_feats[::-1], fontsize=7)
    ax.axvline(0, color='gray', linestyle='--', linewidth=0.8)
    ax.set_xlabel('SHAP 값', fontsize=8)
    ax.set_title(f'{title}\n예측 RevPAR ≈ ₩{shap_pred_orig[idx]:,.0f}', fontsize=9)

plt.suptitle('SHAP Waterfall — RevPAR 수준별 주요 드라이버', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print('저장: reports/fig_03_shap_waterfall.png')

## 호스트 액션 가이드 — SHAP 기반

SHAP 분석 결과를 바탕으로 호스트가 **즉시 실행 가능한 RevPAR 최적화 전략**을 도출합니다.

In [ ]:
# SHAP 중요도를 호스트 관점의 실행 가능한 액션으로 변환

top10 = shap_importance.head(10)

mean_shap_signed = pd.Series(shap_values.mean(axis=0), index=feature_names)
direction = mean_shap_signed[top10.index]

# 호스트 액션 매핑
action_map = {
    'rating_overall':         ('평점 유지', '4.8 이상 평점 유지 — 청결·소통·위치 요인 집중'),
    'photos_count':           ('사진 최적화', '21~35장 구간 목표 — 고품질 사진 우선'),
    'num_reviews':            ('리뷰 축적', '체크아웃 후 즉시 리뷰 요청 메시지 발송'),
    'superhost':              ('슈퍼호스트 달성', '응답률 90%↑, 취소율 최소화, 평점 4.8↑ 유지'),
    'instant_book':           ('즉시예약 활성화', '즉시예약 ON → 점유율 개선 → RevPAR 향상'),
    'min_nights':             ('최소숙박일 최적화', '2~3박 설정 (단기 점유율 극대화)'),
    'extra_guest_fee_policy': ('추가요금 정책', '대형 숙소(large guest) — 추가요금 정책 활성화'),
    'review_rate':            ('리뷰 전환율', '투숙객의 리뷰 전환 유도 (메시지 커스텀)'),
    'photos_tier':            ('사진 등급 관리', '고품질 전문 촬영으로 photos_tier 상향'),
    'nearest_poi_dist_km':    ('입지 강조', '관광명소/교통허브와 거리 → 리스팅에 명시'),
}

print('=' * 70)
print('🏠 호스트 RevPAR 최적화 액션 가이드 (SHAP 기반, 중요도 순)')
print('=' * 70)
for i, (feat, shap_val) in enumerate(top10.items(), 1):
    sign = '▲ +RevPAR' if direction[feat] > 0 else '▼ -RevPAR'
    action_name, action_detail = action_map.get(feat, (feat, '추가 분석 필요'))
    print(f'{i:2d}. [{sign}] {action_name}')
    print(f'    피처: {feat}  |  평균 |SHAP|: {shap_val:.4f}')
    print(f'    실행: {action_detail}')
    print()

print('=' * 70)

In [ ]:
# 예측값 범위 검증 -- 음수 또는 200만원 초과 예측은 도메인 기준 이탈
y_pred_final = lgbm_model.predict(X_te_best)
y_pred_final_orig = np.expm1(y_pred_final)

neg_preds    = (y_pred_final_orig < 0).sum()
over_preds   = (y_pred_final_orig > REVPAR_MAX).sum()
valid_preds  = len(y_pred_final_orig) - neg_preds - over_preds

print('=' * 60)
print('🎯 최종 모델 성능 요약 (LightGBM)')
print('=' * 60)
r = test_results['LightGBM']
print(f"  Test R²   (log):  {r['Test R2 (log)']:.4f}")
print(f"  Test MAE  (₩):   ₩{r['Test MAE (원본₩)']:>12,.0f}")
print(f"  Test RMSE (₩):   ₩{r['Test RMSE (원본₩)']:>12,.0f}")
print(f"  Test MAPE (%):    {r['Test MAPE (%)']:.1f}%")
print()
print('🔍 예측값 범위 검증 (₩0 ~ ₩2,000,000)')
print(f'  총 예측 수:     {len(y_pred_final_orig):,}')
print(f'  정상 범위:      {valid_preds:,} ({valid_preds/len(y_pred_final_orig)*100:.1f}%)')
print(f'  음수 예측:      {neg_preds}')
print(f'  상한 초과:      {over_preds}')
assert neg_preds + over_preds < len(y_pred_final_orig) * 0.01, \
    f'경고: 범위 이탈 예측이 1%를 초과합니다 ({neg_preds+over_preds}개)'
print('✅ 예측값 범위 검증 통과')
print()
print('=' * 60)
print('📁 저장된 시각화 파일 목록:')
for f in sorted(REPORT_DIR.glob('fig_03_*.png')):
    print(f'  {f.name}')
print('=' * 60)